In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle
import warnings
warnings.filterwarnings('ignore')

In [7]:
url = ('../data/raw/playstore_reviews.csv')
try:
    df = pd.read_csv(url)
    print("    Datos cargados correctamente")
except:
    df = pd.read_csv(url, engine='python', on_bad_lines='skip')
    print("    Datos cargados (saltando líneas con errores)")

# Limpiar nombres de columnas
df.columns = df.columns.str.strip()

print(f"   - Total de filas: {df.shape[0]}")
print(f"   - Columnas: {list(df.columns)}")

# Eliminar nulos
df = df.dropna()
print(f"   - Filas después de limpiar nulos: {df.shape[0]}")

    Datos cargados (saltando líneas con errores)
   - Total de filas: 87
   - Columnas: ['package_name', 'review', 'polarity']
   - Filas después de limpiar nulos: 87


In [9]:
print("\n DISTRIBUCIÓN DE CLASES:")
print(df['polarity'].value_counts())

neg = (df['polarity']==0).sum()
pos = (df['polarity']==1).sum()
print(f"   - Negativas (0): {neg} ({neg/len(df)*100:.1f}%)")
print(f"   - Positivas (1): {pos} ({pos/len(df)*100:.1f}%)")


 DISTRIBUCIÓN DE CLASES:
polarity
0    86
1     1
Name: count, dtype: int64
   - Negativas (0): 86 (98.9%)
   - Positivas (1): 1 (1.1%)


# PREPROCESAMIENTO

In [10]:
# Eliminar columna innecesaria
if 'package_name' in df.columns:
    df = df.drop('package_name', axis=1)

# Limpiar texto: minúsculas y espacios
df['review'] = df['review'].astype(str).str.strip().str.lower()

# Eliminar textos muy cortos
df = df[df['review'].str.len() > 10]
print(f"   - Textos procesados: {df.shape[0]}")
print(f"\n   Ejemplo de texto limpio:")
print(f"   '{df['review'].iloc[0][:70]}...'")

   - Textos procesados: 87

   Ejemplo de texto limpio:
   'privacy at least put some option appear offline. i mean for some peopl...'


# DIVIDIR DATOS (TRAIN/TEST)

In [12]:
X = df['review']
y = df['polarity']

# Verificar que tenemos suficientes datos
print(f"   - Total de ejemplos: {len(X)}")
print(f"   - Distribución: {y.value_counts().to_dict()}")

# Dividir: 80% train, 20% test
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print("    División con stratify (mantiene proporción)")
except ValueError:
    # Si stratify falla por pocos datos
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print("    División sin stratify")

print(f"   - Train: {len(X_train)} ejemplos")
print(f"   - Test: {len(X_test)} ejemplos")

   - Total de ejemplos: 87
   - Distribución: {0: 86, 1: 1}
    División sin stratify
   - Train: 69 ejemplos
   - Test: 18 ejemplos


# VECTORIZACIÓN

In [14]:
print("   (Convertir palabras en números)")

# Crear vectorizador con parámetros ajustados
vec = CountVectorizer(
    stop_words='english',  # Elimina palabras comunes (the, is, and...)
    max_features=200,      # Máximo 200 palabras más importantes
    min_df=1,              # Palabra debe aparecer al menos 1 vez
    max_df=0.9             # Ignora palabras muy frecuentes
)

# Entrenar y transformar
print("   Transformando textos...")
X_train_vec = vec.fit_transform(X_train)  # Formato sparse (ahorra memoria)
X_test_vec = vec.transform(X_test)

print(f"    Vocabulario: {len(vec.vocabulary_)} palabras")
print(f"    Matriz train: {X_train_vec.shape}")
print(f"    Matriz test: {X_test_vec.shape}")

# Mostrar algunas palabras del vocabulario
palabras = list(vec.vocabulary_.keys())[:10]
print(f"   Ejemplos de palabras: {palabras}")

   (Convertir palabras en números)
   Transformando textos...
    Vocabulario: 200 palabras
    Matriz train: (69, 200)
    Matriz test: (18, 200)
   Ejemplos de palabras: ['don', 'say', 'updates', 'tell', 'updating', 'old', 'message', 'love', 'able', 'tweets']


# ENTRENAR NAIVE BAYES

In [15]:
resultados = {}

# Modelo 1: Multinomial (MEJOR para texto)
print("\n   A) MULTINOMIAL NAIVE BAYES")
print("      (Cuenta frecuencias de palabras)")
mnb = MultinomialNB()
mnb.fit(X_train_vec, y_train)
y_pred_mnb = mnb.predict(X_test_vec)
acc_mnb = accuracy_score(y_test, y_pred_mnb)
resultados['Multinomial'] = acc_mnb
print(f"      Precisión: {acc_mnb:.4f} ({acc_mnb*100:.2f}%)")

# Modelo 2: Bernoulli
print("\n   B) BERNOULLI NAIVE BAYES")
print("      (Solo ve si la palabra existe o no)")
bnb = BernoulliNB()
bnb.fit(X_train_vec, y_train)
y_pred_bnb = bnb.predict(X_test_vec)
acc_bnb = accuracy_score(y_test, y_pred_bnb)
resultados['Bernoulli'] = acc_bnb
print(f"      Precisión: {acc_bnb:.4f} ({acc_bnb*100:.2f}%)")



   A) MULTINOMIAL NAIVE BAYES
      (Cuenta frecuencias de palabras)
      Precisión: 1.0000 (100.00%)

   B) BERNOULLI NAIVE BAYES
      (Solo ve si la palabra existe o no)
      Precisión: 1.0000 (100.00%)


# MEJOR MODELO:

In [18]:
for modelo, prec in sorted(resultados.items(), key=lambda x: x[1], reverse=True):
    print(f"   {modelo:15s}: {prec:.4f} ({prec*100:.2f}%)")

# Verificar si hay empate
if acc_mnb == acc_bnb:
    if acc_mnb == 1.0:
        print(f"\n   EXCELENTE! Ambos modelos tienen 100% de precision")
        print(f"   Esto es porque:")
        print(f"   - Dataset pequeno con patrones claros")
        print(f"   - Las palabras son muy distintivas")
        print(f"   - La division train/test fue favorable")
    else:
        print(f"\n   EMPATE: Ambos modelos tienen la misma precision")
    
    print(f"\n   Elegimos MULTINOMIAL (mejor para analisis de texto)")
    mejor = 'Multinomial'
    modelo_final = mnb
    y_pred_final = y_pred_mnb
else:
    mejor = max(resultados, key=resultados.get)
    print(f"\n   MEJOR MODELO: {mejor} Naive Bayes")
    modelo_final = mnb if mejor == 'Multinomial' else bnb
    y_pred_final = y_pred_mnb if mejor == 'Multinomial' else y_pred_bnb

   Multinomial    : 1.0000 (100.00%)
   Bernoulli      : 1.0000 (100.00%)

   EXCELENTE! Ambos modelos tienen 100% de precision
   Esto es porque:
   - Dataset pequeno con patrones claros
   - Las palabras son muy distintivas
   - La division train/test fue favorable

   Elegimos MULTINOMIAL (mejor para analisis de texto)


#  REPORTE DETALLADO

In [20]:
print("\nMatriz de Confusion:")
cm = confusion_matrix(y_test, y_pred_final)
print(cm)

# Calcular aciertos y errores
aciertos = accuracy_score(y_test, y_pred_final) * len(y_test)
errores = len(y_test) - aciertos
print(f"\n   Aciertos: {int(aciertos)}/{len(y_test)}")
print(f"   Errores: {int(errores)}/{len(y_test)}")

# Reporte completo
print("\n" + "-" * 60)
print("Reporte de Clasificacion:")
print(classification_report(y_test, y_pred_final))


Matriz de Confusion:
[[18]]

   Aciertos: 18/18
   Errores: 0/18

------------------------------------------------------------
Reporte de Clasificacion:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18

    accuracy                           1.00        18
   macro avg       1.00      1.00      1.00        18
weighted avg       1.00      1.00      1.00        18



# COMPARAR CON REGRESION LOGISTICA

(No incluí Random Forest (lo omití porque crasheaba el kernel))

In [23]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"\n   Regresion Logistica: {acc_lr*100:.2f}%")
print(f"   Naive Bayes:         {resultados[mejor]*100:.2f}%")

# Elegir el mejor para guardar
if acc_lr > resultados[mejor]:
    modelo_guardar = lr
else:
    modelo_guardar = modelo_final

print(f"\n   Modelo a guardar: {'Regresion Logistica' if acc_lr > resultados[mejor] else 'Naive Bayes'}")


   Regresion Logistica: 100.00%
   Naive Bayes:         100.00%

   Modelo a guardar: Naive Bayes


# GUARDAR MODELO

In [25]:
with open('modelo_sentimientos.pkl', 'wb') as f:
    pickle.dump(modelo_guardar, f)
    
with open('vectorizador.pkl', 'wb') as f:
    pickle.dump(vec, f)

print("   OK: Modelo guardado")
print("   OK: Archivos creados:")
print("       - modelo_sentimientos.pkl")
print("       - vectorizador.pkl")

   OK: Modelo guardado
   OK: Archivos creados:
       - modelo_sentimientos.pkl
       - vectorizador.pkl
